# Fine-tuning ModernBERT for Emotion Classification

| Colab | GitHub |
|---|---|
| <a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/bert-fine-tuning-emotion/bert-emotion-tutorial.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> | [Full code and repo](https://github.com/unionai/workshops/tree/main/tutorials/bert-fine-tuning-emotion) |

Fine-tune [ModernBERT](https://huggingface.co/answerdotai/ModernBERT-base) to classify emotions in text, then **explore how the model makes decisions** with attention heatmaps and gradient-based token attribution.

### What we'll build

```
┌──────────┐    ┌────────────┐    ┌────────────┐    ┌─────────────────┐
│ Get Data │───▶│   Train    │───▶│  Evaluate  │───▶│    Explore      │
│  (CPU)   │    │   (GPU)    │    │   (GPU)    │    │   Inference     │
└──────────┘    └────────────┘    └────────────┘    │    (GPU)        │
 emotion         ModernBERT        Confusion        └─────────────────┘
 dataset         fine-tuning       matrix +           Attention heatmaps
                 with live         per-class           + token importance
                 loss/eval         metrics             + misclassification
                 charts                                analysis
```

### What makes this interesting

Beyond just training a classifier, we'll look **inside the model** to understand:
- **Attention heatmaps**: which words does the model "look at" when classifying?
- **Token importance**: which words actually *drive* the prediction? (gradient attribution)
- **Misclassification analysis**: where does the model fail, and why?
- **Negation handling**: the dataset lacks negated examples, so "I am NOT angry" still predicts anger. We'll see why.

Then we'll **deploy the model** as a FastAPI endpoint with a **Gradio frontend** for interactive exploration.

### The dataset

We use [dair-ai/emotion](https://huggingface.co/datasets/dair-ai/emotion), which contains ~20k English Twitter messages labeled with 6 emotions:

| Label | Emotion | Example |
|-------|---------|--------|
| 0 | sadness | "i feel so empty inside" |
| 1 | joy | "i am so happy right now" |
| 2 | love | "i feel blessed to have you" |
| 3 | anger | "i am furious about this" |
| 4 | fear | "i feel so scared and anxious" |
| 5 | surprise | "i cant believe this just happened" |

---

## Setup

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/bert-fine-tuning-emotion
    !uv pip install -r requirements.txt
    !uv pip install keyrings.alt pygments
    !mkdir -p ~/.config/python_keyring && echo -e '[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring' > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

from utils.file_viewer import view_file

### Connect to Flyte cluster

Skip this if you only want to run locally.

- Don't have a cluster? Request demo access at [union.ai](https://union.ai/)
- Or run single node cluster locally with [Flyte Devbox](https://www.union.ai/docs/v2/flyte/user-guide/run-modes/running-devbox/)
- Already have one? Set your endpoint below

In [ ]:
!flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --project workshopbert \
    --domain development \
    --builder remote \
    --auth-type headless

If running on devbox:
```bash
flyte create config \
    --endpoint localhost:30080 \
    --project flytesnacks \
    --domain development \
    --builder local \
    --insecure
```

### Set HuggingFace token (optional)

ModernBERT doesn't require a token, but if you swap to a gated model you'll need one.

In [9]:
# Optional - skip if using ModernBERT (not gated)
# import os
# from getpass import getpass
# os.environ['HF_TOKEN'] = getpass('HF_TOKEN: ')

---

## Run the Pipeline

We'll walk through the code below while our model is training

### Pipeline parameters

| Flag | Default | Description |
|------|---------|-------------|
| `--model_name` | `answerdotai/ModernBERT-base` | HuggingFace encoder model |
| `--epochs` | `3` | Training epochs |
| `--lr` | `2e-5` | Learning rate |
| `--batch_size` | `16` | Batch size |
| `--max_train_samples` | `10000` | Training examples |
| `--max_eval_samples` | `2000` | Eval examples |
| `--num_eval_examples` | `200` | Examples for base vs fine-tuned comparison |
| `--num_explore_examples` | `12` | Examples for attention/attribution analysis |

### Remote run (on Flyte cluster)

In [ ]:
!flyte run workflow.py pipeline \
    --epochs 3 \
    --max_train_samples 4000 \
    --num_eval_examples 200 \
    --num_explore_examples 12

### Local run (no cluster needed)

Add `--local` to run everything on your machine. Smaller dataset for speed:

```bash
flyte run --local --tui workflow.py pipeline \
    --max_train_samples 200 \
    --max_eval_samples 50 \
    --epochs 1 \
    --num_eval_examples 30 \
    --num_explore_examples 6
```

---

## Code Walkthrough

Let's look at the key files before we run anything. The pipeline has 4 tasks chained together, each with rich visual reports.

### Project config

Two environments: **GPU** for training/inference, **CPU** for data prep and orchestration. The image is built from `requirements.txt` and shared across both.

In [3]:
view_file("config.py")

In [4]:
view_file("requirements.txt")

### The workflow

The full pipeline is in `workflow.py`. Let's look at each task.

In [13]:
view_file("workflow.py")

#### Key things to notice in the workflow:

**Training task (`train`):**
- Uses `AutoModelForSequenceClassification` with `num_labels=6`
- Live training report with loss curves, eval accuracy/F1 charts, progress bar
- `ReportCallback` hooks into HuggingFace Trainer to push updates to Flyte UI
- Eval runs at each epoch boundary so you can watch accuracy improve

**Evaluation task (`evaluate`):**
- Compares base model (random classifier head) vs fine-tuned
- Generates confusion matrix heatmap, per-class precision/recall/F1
- Per-class accuracy bar chart (base vs fine-tuned)

**Explore inference task (`explore_inference`):**
- Loads model with `attn_implementation="eager"` to extract attention weights (flash attention doesn't return them)
- **Attention heatmap**: CLS token attention from the last transformer layer, averaged across heads
- **Token importance**: Gradient x embedding attribution, hooks into the embedding layer to compute gradients
- **Misclassification spotlight**: Sorts wrong predictions by confidence to find blind spots

### Report helpers

All visualizations are pure SVG/HTML with no matplotlib dependency. This includes:
- Line charts (training loss, eval metrics)
- Bar charts (per-class accuracy comparison)
- Confusion matrix heatmap
- Colored text for attention/importance visualization
- Confidence bars for emotion scores

In [ ]:
view_file("report_helpers.py")

---

## Run the Pipeline

### Pipeline parameters

| Flag | Default | Description |
|------|---------|-------------|
| `--model_name` | `answerdotai/ModernBERT-base` | HuggingFace encoder model |
| `--epochs` | `3` | Training epochs |
| `--lr` | `2e-5` | Learning rate |
| `--batch_size` | `16` | Batch size |
| `--max_train_samples` | `10000` | Training examples |
| `--max_eval_samples` | `2000` | Eval examples |
| `--num_eval_examples` | `200` | Examples for base vs fine-tuned comparison |
| `--num_explore_examples` | `12` | Examples for attention/attribution analysis |

### Remote run (on Flyte cluster)

In [ ]:
!flyte run workflow.py pipeline \
    --epochs 3 \
    --max_train_samples 4000 \
    --num_eval_examples 200 \
    --num_explore_examples 12

### Local run (no cluster needed)

Add `--local` to run everything on your machine. Smaller dataset for speed:

```bash
flyte run --local --tui workflow.py pipeline \
    --max_train_samples 200 \
    --max_eval_samples 50 \
    --epochs 1 \
    --num_eval_examples 30 \
    --num_explore_examples 6
```

---

## Understanding the Results

While the pipeline runs, check the Flyte UI for live reports. Here's what to look for:

### Training report
- **Loss curve**: should decrease steadily. If it plateaus early, the model may need more data or a different learning rate.
- **Eval accuracy/F1**: updates after each epoch. Expect ~90%+ accuracy on emotion classification with ModernBERT.
- **Progress bar**: shows step/epoch progress with current loss.

### Evaluation report
- **Confusion matrix**: look at which emotions get confused. Common: anger↔fear, love↔joy.
- **Per-class metrics**: some emotions (joy, sadness) are easier to classify than others (love, surprise).
- **Base vs fine-tuned bar chart**: the base model has a random classifier head, so ~16.7% accuracy (1/6). Fine-tuned should be dramatically better.

### Explore inference report
- **Attention heatmaps**: darker tokens = more attention from [CLS]. The model should focus on emotional words ("happy", "terrified", "love").
- **Token importance**: green = supports prediction, red = opposes. Look for cases where function words unexpectedly matter.
- **Misclassification spotlight**: the most confident wrong predictions reveal the model's blind spots.

---

## How Attention Visualization Works

BERT-style models use **multi-head self-attention** where each token attends to every other token, producing an attention weight matrix. Here's what we extract:

```
Input:  [CLS] i am so happy right now [SEP]
                                                    
Last layer attention (averaged across 12 heads):

[CLS] → i(0.05) am(0.08) so(0.15) happy(0.45) right(0.04) now(0.12) [SEP](0.11)
                                    ^^^^^
                            highest attention = most relevant for classification
```

The **[CLS] token** is special. Its final representation is fed to the classifier head. So the [CLS] attention pattern tells us: *"what did the model look at when making its classification decision?"*

### Why `attn_implementation="eager"`?

ModernBERT uses **Flash Attention** by default for speed, but Flash Attention doesn't return attention weight matrices. We switch to eager (standard) attention for the explore_inference task to extract the weights. This is slower but only runs on a few examples.

### Gradient-based token importance

Attention shows where the model *looks*, but not necessarily what *drives* the prediction. For that, we use gradient attribution:

```python
importance(token) = ||grad(prediction, embedding(token)) * embedding(token)||
```

This measures how much each token's embedding *actually influences* the predicted class score. It's complementary to attention. Sometimes a token gets high attention but low importance (the model looked but didn't use it).

---

## Serve the Model

Once the pipeline completes, deploy the fine-tuned model as a live API.

### Step 1: Deploy the FastAPI server

The server loads the model from the pipeline output and exposes a `/predict` endpoint that returns:
- Predicted emotion + confidence
- Full probability distribution across all 6 emotions
- Attention weights per token (for heatmap visualization)

In [ ]:
# Deploy the model server (uses latest pipeline output)
!python serve.py

# Or deploy from a specific run:
# !python serve.py --run-name <run-name-from-flyte-ui>

In [11]:
view_file("serve.py")

### Test the endpoint

In [ ]:
%%bash
# Replace with your deployed URL
curl -X POST https://your-app-url/predict \
  -H "Content-Type: application/json" \
  -d '{"text": "I am so happy today!"}'

### Step 2: Deploy the Gradio frontend

The Gradio app is a thin HTTP client that calls the FastAPI server. It auto-discovers the server endpoint.

In [ ]:
!python app_gradio.py

In [12]:
view_file("app_gradio.py")

---

## Try It: Explore the Model's Behavior

Once the Gradio app is running, try these inputs and observe the attention patterns:

### Straightforward emotions
These should get high confidence with attention focused on the emotional keywords:
- `"I am so happy right now"` → joy, attention on "happy"
- `"This makes me furious"` → anger, attention on "furious"
- `"I love you more than anything"` → love, attention on "love"

### Negation (a dataset gap)
Try these and watch the model fail:
- `"this does not make me angry"` → predicts anger (~99%!)
- `"I am not sad anymore"` → likely predicts sadness
- `"I'm not surprised at all"` → likely predicts surprise

Look at the attention heatmap. The model attends to "angry", "sad", "surprised" but doesn't properly handle the negation. This isn't a limitation of the model architecture itself. The training data (Twitter messages) is mostly direct emotional statements and rarely contains negated examples, so the model simply pattern-matches on emotional keywords. A dataset with more negated examples would teach the model to handle these correctly.

### Ambiguous / mixed emotions
These are interesting because the confidence distribution spreads across multiple emotions:
- `"I can't believe they did that to me"` → anger or surprise?
- `"I'm nervous but excited about tomorrow"` → fear or joy?
- `"I miss you so much"` → sadness or love?

### Subtle / complex
- `"The meeting went fine"` → low confidence, ambiguous
- `"Whatever"` → what does the model think apathy looks like?
- `"I just got promoted but my best friend got laid off"` → mixed signals

---

## Why ModernBERT?

[ModernBERT](https://huggingface.co/answerdotai/ModernBERT-base) (2024) is a drop-in replacement for BERT-base with modern improvements:

| Feature | BERT (2018) | ModernBERT (2024) |
|---------|-------------|-------------------|
| Context length | 512 tokens | 8,192 tokens |
| Position encoding | Absolute | Rotary (RoPE) |
| Attention | Standard | Flash Attention |
| Parameters | ~110M | ~150M |
| Training data | BooksCorpus + Wikipedia | 2T tokens, diverse sources |

Same `AutoModelForSequenceClassification` API, better results. The attention visualization works identically since it's still a multi-head transformer encoder.

You can swap to classic BERT anytime:
```bash
flyte run workflow.py pipeline --model_name "bert-base-uncased"
```

---

## Key Takeaways

**Fine-tuning:**
- Encoder models (BERT/ModernBERT) are excellent for classification. Fast to train, small to deploy.
- `AutoModelForSequenceClassification` + HuggingFace `Trainer` handles most of the complexity
- A random classifier head gets ~16% accuracy. A few epochs of fine-tuning gets 90%+.

**Model interpretability:**
- Attention heatmaps show *where* the model looks (CLS token attention pattern)
- Gradient attribution shows *what drives* the prediction (which tokens influence the score)
- These are complementary. A token can have high attention but low importance.
- Misclassification analysis reveals systematic blind spots (negation, ambiguity)

**Flyte/Union:**
- Pipeline structure (get_data → train → evaluate → explore) is reusable for any classification task
- Live reports make training observable without TensorBoard
- `flyte.io.Dir` passes model artifacts between tasks seamlessly
- Same model serves in production via FastAPI + Gradio with `RunOutput` parameter binding

---

## Resources

- Full code: [tutorials/bert-fine-tuning-emotion](https://github.com/unionai/workshops/tree/main/tutorials/bert-fine-tuning-emotion)
- Get started with Flyte: [union.ai/docs](https://www.union.ai/docs/v2/flyte/user-guide/run-modes/)
- Book a consultation: [union.ai/consultation](https://www.union.ai/consultation)
- Join the Slack: [slack.flyte.org](https://slack.flyte.org/)
- More live events: [luma.com/unionai](https://luma.com/unionai)